# Phase 1: Dataset Audit & Exploratory Data Analysis
## AI-Based Early Warning System for Low Dissolved Oxygen in Fish Farms

### Project Overview
In aquaculture ponds, dissolved oxygen (DO) is the single most critical parameter for fish survival and growth. When DO drops below critical levels (such as 3.0 mg/L), fish experience severe hypoxial stress, which can lead to rapid mortality within hours.

**Ultimate Project Goal:** Predict whether a fish-farm pond will fall below **3.0 mg/L dissolved oxygen** within the next **2 hours**.

**Phase 1 Scope:**
- Set up the clean environment and inspect the 17 pond continuous monitoring CSV files from the FWI Continuous Water Quality Monitoring Dataset.
- Validate timestamp continuity, expected 15-minute intervals, and data gaps.
- Audit Quality Control (QC) flags and equipment artifact zero-readings.
- Perform a rigorous feasibility check for the provisional 3.0 mg/L threshold.

In [ ]:
# 1. Environment & Module Setup
import sys
from pathlib import Path

# Ensure project root is accessible
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import find_csv_files, load_all_ponds, combine_pond_data
from src.data_quality import (
    audit_all_ponds,
    audit_combined_dataset,
    summarize_qc_by_pond,
    feasibility_summary_all_ponds,
)

print("Environment initialized and modules loaded successfully.")

### 2. Discover CSV Files
We scan `data/raw/csv/` to locate all CSV files and verify that 17 continuous monitoring pond files and 5 metadata/reference files are identified.

In [ ]:
raw_dir = project_root / "data" / "raw" / "csv"
pond_files, meta_files = find_csv_files(raw_dir)

print(f"Found {len(pond_files)} pond time-series CSV files.")
print(f"Found {len(meta_files)} metadata/reference CSV files.")
for idx, pf in enumerate(pond_files, 1):
    print(f"  {idx:02d}. {pf.name}")

### 3. Load All Ponds and Combine into Unified DataFrame
Timestamps are parsed, source files tracked, and original sensor readings preserved.

In [ ]:
dfs = load_all_ponds(raw_dir)
combined_df = combine_pond_data(dfs)

print(f"Total combined rows: {len(combined_df):,}")
print(f"Unique ponds: {combined_df['pond_id'].nunique()}")
print(f"Columns: {list(combined_df.columns)}")
combined_df.head()

### 4. Pond-by-Pond Descriptive Statistics & Quality Audit
Calculate rows, duration, sampling interval adherence, recording gaps, and sensor summary statistics.

In [ ]:
pond_audit = audit_all_ponds(dfs)
display_cols = [
    "pond_id", "rows", "duration_days", "pct_close_15min",
    "gaps_gt_20min", "zero_do", "zero_ph", "zero_temp",
    "mean_do", "mean_ph", "mean_temp"
]
pond_audit[display_cols]

### 5. Quality Control (QC) Flags & Sensor Artifacts
Audit occurrences of QC flags (such as time gaps, abrupt DO jumps, and pH out of range) and zero-value readings.

In [ ]:
qc_summary = summarize_qc_by_pond(combined_df)
qc_summary

### 6. DO < 3.0 mg/L Feasibility Analysis
Evaluate whether low-DO occurrences provide a viable distribution for training an ML model in Phase 2.
Sensor initialization artifacts (DO = 0.0 mg/L) are excluded from the valid biological observation counts.

In [ ]:
feasibility = feasibility_summary_all_ponds(dfs, threshold=3.0)
total_valid_low = feasibility["valid_below_3_count"].sum()
print(f"Total valid DO < 3.0 mg/L observations: {total_valid_low:,}")
print(f"Ponds experiencing DO < 3.0 mg/L: {(feasibility['valid_below_3_count'] > 0).sum()} of {len(feasibility)}")
feasibility

### 7. Visualizations
Plot sensor distributions, observation counts, and low-DO occurrences.

In [ ]:
# Figure 1: Dissolved Oxygen Distribution
plt.figure(figsize=(8, 4))
plt.hist(combined_df["do_mg_l"].dropna(), bins=50)
plt.title("DO Distribution - All Ponds Combined")
plt.xlabel("DO (mg/L)")
plt.ylabel("Frequency (Count)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Figure 2: Valid DO < 3.0 mg/L Observations by Pond
plt.figure(figsize=(10, 5))
plt.barh(feasibility["pond_id"], feasibility["valid_below_3_count"])
plt.title("Valid DO Observations Below 3.0 mg/L by Pond")
plt.xlabel("Count of Observations")
plt.ylabel("Pond ID")
plt.grid(True, linestyle="--", alpha=0.5, axis="x")
plt.tight_layout()
plt.show()

### 8. Phase 1 Key Findings & Recommendations
- **Data Scale:** 17 ponds, 72,750 observations covering November 2025 to January 2026.
- **Target Feasibility:** All 17 ponds show valid readings below 3.0 mg/L, totaling 15,451 observations (21.24% of the dataset). This proves high feasibility for predictive modeling.
- **Data Quality Considerations for Phase 2:**
  1. Exact zeros (DO=0, pH=0, Temp=0) must be treated as equipment artifacts and filtered before modeling.
  2. Duplicate timestamps exist in `ara2_3ed4df8e` and `ara2_6ca69422` and must be systematically deduplicated.
  3. Large recording gaps (>20 min, multi-day harvest uninstallation) must be respected so that time-series lags and rolling windows do not bridge across sensor outages.